In [1]:
import json

data_path = './deita_sota_pool/instag_mix.json'
data = [json.loads(line) for line in open(data_path)]

In [2]:
len(data)

306171

In [3]:
data[0]

{'id': 'LbHON3Q',
 'conversations': [{'from': 'human',
   'value': 'People may now think of queries and replies in novel and creative ways thanks to new AI text systems. What are some crucial inquiries we may pose to these text-based AIs that would yield original and admirable insights into humanity? Make six specific questions that don’t contain the phrase “AI.”'},
  {'from': 'gpt',
   'value': '1. What is the meaning of life and how does it differ from individual to individual?\n2. How does our understanding of the world shape our beliefs and values?\n3. What are the most important qualities that make us human?\n4. How do our emotions influence our decision-making processes and behavior?\n5. What is the relationship between our thoughts and actions, and how does it impact our daily lives?\n6. Can humans find happiness and fulfillment by pursuing material wealth, or is there something more to life?'},
  {'from': 'human', 'value': 'Continue writing please'},
  {'from': 'gpt',
   'value

In [4]:
len_list =[]
for item in data:
    len_list.append(item['source'])
    
set(len_list)

{'ShareGPT', 'UltraChat', 'WizardLM_evol_instruct_V2'}

In [5]:
# source_list = ['ShareGPT', 'WizardLM_evol_instruct_V2', 'UltraChat']
source_dict = {
    'ShareGPT' : [],
    'WizardLM_evol_instruct_V2' : [],
    'UltraChat' : []
}

for idx,item in enumerate(data):
    source = item['source']
    source_dict[source].append(idx)

In [6]:
# role_list = ['human', 'gpt', 'system', 'bing', 'chatgpt', 'bard', 'user']
role_dict = {
    'human' : [],
    'gpt' : [],
    'system' : [],
    'bing' : [],
    'chatgpt' : [],
    'bard' : [],
    'user' : []
}
for idx,item in enumerate(data):
    conversations_list = item['conversations']
    for cov in conversations_list:
        role_dict[cov['from']].append(idx)

In [7]:
role_dict['system'][1]

874

In [14]:
data[181]

{'id': 'gOGlcwj',
 'conversations': [{'from': 'system',
   'value': '*This chat conversation is shared from [**TypingMind.com**](https://typingmind.com)*'},
  {'from': 'human',
   'value': 'Fix all the grammar errors in the text below. Only fix grammar errors, do not change the text style.\n\nI love AI a very many'},
  {'from': 'gpt', 'value': 'I love AI very much.'}],
 'source': 'ShareGPT'}

In [8]:
'''
processed deita_sota
convert to sharegpt format
reference: https://github.com/hiyouga/LLaMA-Factory/blob/5177f3ba90f369ec55bb270d5fb868f1b94e3acf/data/README.md
'''
def check_adjacent_elements(lst):
    for i in range(len(lst) - 1):
        if lst[i] == lst[i+1]:
            return False
    return True

def process_data(conversations: list):
    conv_role_list = []
    updated_conversations = []

    while conversations[0]['from'] == 'system':
        conversations = conversations[1:]
        if len(conversations) < 1:
            return []

    while conversations[0]['from'] == 'gpt':
        conversations = conversations[1:]
        if len(conversations) < 1:
            return []
        
    while conversations[0]['from'] == 'bing':
        conversations = conversations[1:]
        if len(conversations) < 1:
            return []

    while conversations[0]['from'] == 'chatgpt':
        conversations = conversations[1:]
        if len(conversations) < 1:
            return []

    while conversations[0]['from'] == 'bard':
        conversations = conversations[1:]
        if len(conversations) < 2:
            return []

    while conversations[-1]['from'] == 'human':
        conversations = conversations[:-1]
        if len(conversations) < 2:
            return []
        
    while conversations[-1]['from'] == 'user':
        conversations = conversations[:-1]
        if len(conversations) < 2:
            return []
        
    for conversation in conversations:
        conv_role_list.append(conversation['from'])
        
        
    flag = check_adjacent_elements(conv_role_list)
    
    # 如果有的conv='' flag=False, eg: {'from': 'gpt', 'value': ''}
    for conversation in conversations:
        if conversation['value'] == '':
            flag = False
    
    if flag:
        # 如果不存在连续相同的角色，falg=True
        for conversation in conversations:
            updated_conversation = {}
            if conversation['from'] in ['bing', 'chatgpt', 'bard']:
                updated_conversation['from'] = 'gpt'
                updated_conversation['value'] = conversation['value']
            elif conversation['from'] in ['user']:
                updated_conversation['from'] = 'human'
                updated_conversation['value'] = conversation['value']
            elif conversation['from'] == 'human':
                updated_conversation['from'] = 'human'
                updated_conversation['value'] = conversation['value']
            elif conversation['from'] == 'gpt':
                updated_conversation['from'] = 'gpt'
                updated_conversation['value'] = conversation['value']
            elif conversation['from'] == 'system':
                continue
            else:
                print(conversations)
            updated_conversations.append(updated_conversation)
            

    else:
        updated_conversations = []
    return updated_conversations

updated_conversations_list = []
noise_list = []

for idx,item in enumerate(data):
    item_dict = {}
    item_dict['id'] = item['id']
    item_dict['source'] = item['source']
    if process_data(item['conversations']) == []:
        noise_list.append(idx)
        continue
    item_dict['conversations'] = process_data(item['conversations'])
    updated_conversations_list.append(item_dict)

In [9]:
len(updated_conversations_list)

305566

In [12]:
import json
filename = './deita_sota_pool/deita_sota_pool_305566.json'
with open(filename, 'w') as f:
    json.dump(updated_conversations_list, f)

In [17]:
'''
Count turns that are not even
'''
noise_len_conversations_list = []
for idx,item in enumerate(updated_conversations_list):
    if len(item['conversations'])  % 2 != 0 or len(item['conversations'])==0:
        noise_len_conversations_list.append(idx)
        
len(noise_len_conversations_list)

0